# 🧪 Lasmoid 10M — Quick Validation Run

**Train a tiny 10M model in ~30 minutes to verify the full pipeline produces real output (not gibberish).**

This is NOT for production — it's a fast sanity check that:
- ✅ All subsystems work (attention, SSM, mHC, MoE, concept memory, MTP)
- ✅ Loss decreases consistently
- ✅ Generated text is coherent English (not random tokens)
- ✅ The full pretrain→SFT→generate pipeline runs end-to-end

**Key insight**: 10M params CAN produce real English if trained long enough on clean data.
The trick is using TinyStories (simple vocabulary, short sentences) rather than web text.
A 10M model trained on 50K steps of TinyStories will generate coherent children's stories.

| Spec | Value |
|------|-------|
| Params | ~10M |
| VRAM | ~2GB |
| Train time | ~30min (Kaggle/Colab/CPU) |
| Dataset | TinyStories (simple English) |
| Steps | 5000 pretrain + 1000 SFT |

---

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1: Setup
# ═══════════════════════════════════════════════════════════════════
!pip install -q torch transformers safetensors tiktoken numpy tqdm datasets huggingface_hub

import os, sys, json, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm

IN_KAGGLE = os.path.exists('/kaggle')
WORK_DIR = '/kaggle/working/Lasmoid' if IN_KAGGLE else '/content/Lasmoid'

# Use GPU if available, CPU works too for 10M
if torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
    print('✅ Apple MPS')
else:
    DEVICE = 'cpu'
    print('⚠️  CPU only (still works for 10M, just slower)')

print(f'PyTorch {torch.__version__}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2: Load Lasmoid Code
# ═══════════════════════════════════════════════════════════════════
# Upload/clone your Lasmoid code first, then:
# !cp -r /kaggle/input/lasmoid-code/Lasmoid {WORK_DIR}
# OR: !git clone https://github.com/Theory903/Lasmoid.git {WORK_DIR}

assert os.path.exists(os.path.join(WORK_DIR, 'inference', 'model.py')), \
    f'❌ Lasmoid not at {WORK_DIR} — clone or upload it first!'

sys.path.insert(0, WORK_DIR)
sys.path.insert(0, os.path.join(WORK_DIR, 'inference'))
sys.path.insert(0, os.path.join(WORK_DIR, 'train'))
os.chdir(WORK_DIR)
print(f'✅ Lasmoid loaded')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3: Build 10M Model
# ═══════════════════════════════════════════════════════════════════
from dataclasses import fields
import transformers
from inference.model import Lasmoid, ModelArgs, compute_loss
from train.optimizer import Muon, build_optimizers
from train.scheduler import WSDScheduler

with open('config_10m.json') as f:
    cfg = json.load(f)

valid = {f.name for f in fields(ModelArgs)}
args = ModelArgs(**{k: v for k, v in cfg.items() if k in valid})

# Tokenizer
enc = transformers.PreTrainedTokenizerFast.from_pretrained(WORK_DIR, fix_mistral_regex=True)
args.vocab_size = max(args.vocab_size, len(enc))
EOS_ID = enc.eos_token_id or 1
SEQ_LEN = args.max_seq_len  # 256

torch.manual_seed(42)
model = Lasmoid(args).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'\n🏗️  Lasmoid-10M: {n_params:.1f}M params')
print(f'    dim={args.dim}, layers={args.n_layers}, heads={args.n_heads}')
print(f'    experts={args.n_routed_experts}, streams={args.num_residual_streams}')
print(f'    VRAM: ~{sum(p.numel()*2 for p in model.parameters())/1e9:.1f}GB (bf16)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4: Download TinyStories (best for small models — real English)
# ═══════════════════════════════════════════════════════════════════
from datasets import load_dataset

# TinyStories: synthetic children's stories with simple grammar.
# Even a 10M model can learn to generate coherent English from this.
# This is WHY this dataset — web text needs 100M+ params to be coherent.
print('📥 Downloading TinyStories (the best dataset for small model coherence)...')
ds = load_dataset('roneneldan/TinyStories', split='train')

# Use 20K stories — enough for 5K steps at batch=8×seq=256
N_STORIES = 30000
ds = ds.select(range(min(N_STORIES, len(ds))))

print(f'Tokenizing {len(ds)} stories...')
all_tokens = []
for ex in tqdm(ds, desc='Tokenize'):
    text = ex.get('text', '')
    if text and len(text) > 20:
        tokens = enc.encode(text) + [EOS_ID]
        all_tokens.extend(tokens)

# Pack
all_tokens = all_tokens[:len(all_tokens) - len(all_tokens) % SEQ_LEN]
data = torch.tensor(all_tokens, dtype=torch.long)
n_train = int(0.95 * len(data))
train_data = data[:n_train]
val_data = data[n_train:]

print(f'\n📊 Data: {len(train_data)/1e6:.1f}M train tokens, {len(val_data)/1e6:.1f}M val tokens')
del all_tokens, ds

BATCH = 8  # 10M is tiny — use large batch for stability

def get_batch(split='train'):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - SEQ_LEN - 1, (BATCH,))
    x = torch.stack([d[i:i+SEQ_LEN] for i in ix]).to(DEVICE)
    y = torch.stack([d[i+1:i+SEQ_LEN+1] for i in ix]).to(DEVICE)
    return x, y, torch.ones_like(x, dtype=torch.float32)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5: PRETRAIN (5000 steps, ~15-20 min on T4, ~30 min on CPU)
# ═══════════════════════════════════════════════════════════════════
STEPS = 5000

opts = build_optimizers(model, muon_lr=3e-3, adamw_lr=5e-4, weight_decay=0.1)
warmup = int(0.03 * STEPS)
stable = int(0.87 * STEPS)
decay = STEPS - warmup - stable
scheduler = WSDScheduler(
    opts, warmup_steps=warmup, stable_steps=stable, decay_steps=decay,
    base_lrs=[[g['lr'] for g in opt.param_groups] for opt in opts],
    min_lr_ratio=0.1,
)

print(f'🚀 Pretraining: {STEPS} steps, batch={BATCH}, seq={SEQ_LEN}')
print(f'   Tokens per step: {BATCH * SEQ_LEN:,} | Total: ~{STEPS * BATCH * SEQ_LEN / 1e6:.0f}M tokens')
print('─' * 60)

model.train()
losses = []
t0 = time.time()

for step in range(STEPS):
    scheduler.step(step)
    for opt in opts:
        opt.zero_grad(set_to_none=True)
    
    xb, yb, mask = get_batch()
    
    if DEVICE == 'cuda':
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            logits, mtp_logits, _, _, rmaps, _, adjs, eprobs = model(xb, xb)
            loss = compute_loss(logits, yb, rmaps, [model.last_vq_loss], adjs, eprobs,
                                loss_mask=mask, moe_aux_loss=model.last_moe_loss, ignore_index=-100)
            if mtp_logits is not None:
                loss = loss + 0.3 * F.cross_entropy(
                    mtp_logits.view(-1, args.vocab_size), yb[:, 1:].contiguous().view(-1), ignore_index=-100)
    else:
        logits, mtp_logits, _, _, rmaps, _, adjs, eprobs = model(xb, xb)
        loss = compute_loss(logits, yb, rmaps, [model.last_vq_loss], adjs, eprobs,
                            loss_mask=mask, moe_aux_loss=model.last_moe_loss, ignore_index=-100)
        if mtp_logits is not None:
            loss = loss + 0.3 * F.cross_entropy(
                mtp_logits.view(-1, args.vocab_size), yb[:, 1:].contiguous().view(-1), ignore_index=-100)
    
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    for opt in opts:
        opt.step()
    
    losses.append(loss.item())
    
    if step % 200 == 0 and step > 0:
        avg = sum(losses[-200:]) / 200
        elapsed = time.time() - t0
        tps = step * BATCH * SEQ_LEN / elapsed
        eta = (STEPS - step) / (step / elapsed) / 60
        print(f'  Step {step:4d}/{STEPS} | Loss {avg:.3f} | {tps:.0f} tok/s | ETA {eta:.0f}min')

elapsed = (time.time() - t0) / 60
print(f'\n✅ Pretrain done! Final loss: {losses[-1]:.3f} | Time: {elapsed:.0f}min')
print(f'   Loss went from {losses[0]:.2f} → {sum(losses[-50:])/50:.2f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6: Plot loss curve
# ═══════════════════════════════════════════════════════════════════
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 3))
    w = 50
    smoothed = [sum(losses[max(0,i-w):i+1])/len(losses[max(0,i-w):i+1]) for i in range(len(losses))]
    plt.plot(smoothed, linewidth=1)
    plt.xlabel('Step'); plt.ylabel('Loss'); plt.title('Lasmoid-10M Pretrain Loss')
    plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
except:
    print('(matplotlib not available, skipping plot)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7: SFT on story prompts (1000 steps — teaches instruction following)
# ═══════════════════════════════════════════════════════════════════

# For 10M, simple story prompts work better than complex instruction data.
# We format TinyStories as "prompt → continuation" pairs.
print('📝 Preparing SFT data (story prompt → completion)...')

sft_data = []
# Reload a fresh batch of stories for SFT formatting
ds_sft = load_dataset('roneneldan/TinyStories', split='train')
ds_sft = ds_sft.select(range(N_STORIES, min(N_STORIES + 5000, len(ds_sft))))

for ex in ds_sft:
    text = ex.get('text', '').strip()
    if len(text) < 50:
        continue
    # Split story into prompt (first sentence) + completion (rest)
    sentences = text.split('. ')
    if len(sentences) < 2:
        continue
    prompt = sentences[0] + '.'
    completion = '. '.join(sentences[1:])
    
    p_ids = enc.encode(f'Story: {prompt}\nContinue: ')
    c_ids = enc.encode(completion) + [EOS_ID]
    full = p_ids + c_ids
    
    if len(full) > SEQ_LEN:
        full = full[:SEQ_LEN]
        split_idx = min(len(p_ids), SEQ_LEN - 1)
    else:
        split_idx = len(p_ids)
        full = full + [EOS_ID] * (SEQ_LEN - len(full))
    
    mask = [0.0] * split_idx + [1.0] * (len(c_ids)) + [0.0] * (SEQ_LEN - split_idx - len(c_ids))
    mask = mask[:SEQ_LEN]
    sft_data.append((full[:SEQ_LEN], mask))

print(f'   {len(sft_data)} SFT examples ready')

def get_sft_batch():
    idx = torch.randint(len(sft_data), (BATCH,))
    x_list, m_list = [], []
    for i in idx:
        t, m = sft_data[i.item()]
        x_list.append(torch.tensor(t, dtype=torch.long))
        m_list.append(torch.tensor(m, dtype=torch.float32))
    x = torch.stack(x_list).to(DEVICE)
    y = torch.cat([x[:, 1:], torch.full((x.shape[0], 1), EOS_ID, dtype=torch.long, device=DEVICE)], dim=1)
    return x, y, torch.stack(m_list).to(DEVICE)

# SFT training
SFT_STEPS = 1000
sft_opts = build_optimizers(model, muon_lr=1e-3, adamw_lr=2e-4, weight_decay=0.01)

print(f'\n🎓 SFT: {SFT_STEPS} steps')
print('─' * 60)
model.train()
t0 = time.time()

for step in range(SFT_STEPS):
    for opt in sft_opts:
        opt.zero_grad(set_to_none=True)
    xb, yb, mask = get_sft_batch()
    
    if DEVICE == 'cuda':
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            logits, _, _, _, rmaps, _, adjs, eprobs = model(xb, xb)
            loss = compute_loss(logits, yb, rmaps, [model.last_vq_loss], adjs, eprobs,
                                loss_mask=mask, moe_aux_loss=model.last_moe_loss, ignore_index=-100)
    else:
        logits, _, _, _, rmaps, _, adjs, eprobs = model(xb, xb)
        loss = compute_loss(logits, yb, rmaps, [model.last_vq_loss], adjs, eprobs,
                            loss_mask=mask, moe_aux_loss=model.last_moe_loss, ignore_index=-100)
    
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    for opt in sft_opts:
        opt.step()
    
    if step % 200 == 0 and step > 0:
        print(f'  SFT Step {step}/{SFT_STEPS} | Loss {loss.item():.3f}')

print(f'✅ SFT done! Time: {(time.time()-t0)/60:.0f}min')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8: GENERATE TEXT — Verify it's real English!
# ═══════════════════════════════════════════════════════════════════
from inference.sampler import full_sample

model.eval()

@torch.no_grad()
def generate(prompt, max_tokens=100, temperature=0.8, min_p=0.05):
    tokens = enc.encode(prompt)
    generated = list(tokens)
    
    # Pad to seq_len and run prefill
    pad_len = SEQ_LEN - len(tokens)
    if pad_len > 0:
        idx = torch.tensor([([EOS_ID] * pad_len) + tokens], dtype=torch.long, device=DEVICE)
    else:
        idx = torch.tensor([tokens[-SEQ_LEN:]], dtype=torch.long, device=DEVICE)
    
    logits, *_ = model(idx, idx, start_pos=0)
    
    gen = torch.Generator(device='cpu').manual_seed(42)
    for i in range(max_tokens):
        next_logits = logits[:, -1, :].cpu().float()
        next_id = full_sample(next_logits, generated, temperature=temperature,
                              min_p=min_p, generator=gen)
        tid = next_id.item()
        if tid == EOS_ID:
            break
        generated.append(tid)
        
        inp = torch.tensor([[tid]], dtype=torch.long, device=DEVICE)
        logits, *_ = model(x_enc=None, x_dec=inp, start_pos=SEQ_LEN + i)
    
    return enc.decode(generated[len(tokens):])

# ── Test prompts (simple, matching TinyStories training distribution) ──
prompts = [
    'Once upon a time, there was a little girl named',
    'The dog ran to the park and',
    'One day, a boy found a magic',
    'Story: The cat was very hungry.\nContinue:',
    'Lily wanted to play with her',
]

print('\n📝 Generated Text (should be coherent simple English):')
print('═' * 60)
for p in prompts:
    out = generate(p, max_tokens=80, temperature=0.7)
    print(f'\n  Prompt: "{p}"')
    print(f'  Output: {out[:200]}')
    print('─' * 60)

print('\n🔍 If the output is real English sentences (even simple ones), the model works!')
print('   If it\'s gibberish, increase STEPS to 10000 or use more training data.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9: Validation — Check all subsystems are active
# ═══════════════════════════════════════════════════════════════════
print('\n🔬 Subsystem Validation:')
print('─' * 40)

model.eval()
xb, yb, _ = get_batch('val')
with torch.no_grad():
    logits, mtp_logits, concept_db, memory_state, rmaps, indices, adjs, eprobs = model(xb, xb)

# Check each subsystem
checks = {
    'Attention + SSM': logits is not None and logits.shape[-1] == args.vocab_size,
    'MTP (t+2)': mtp_logits is not None,
    'Concept Memory': concept_db is not None and concept_db.shape[-1] == args.dim,
    'MoE Routing': len(rmaps) > 0 and rmaps[0].shape[0] == BATCH,
    'VQ Indices': indices is not None,
    'GVQ Adjacency': len(adjs) > 0,
    'CIF Events': eprobs is not None and len(eprobs) > 0,
    'mHC Streams': args.num_residual_streams == 2,
    'Loss finite': torch.isfinite(model.last_vq_loss),
}

all_pass = True
for name, ok in checks.items():
    status = '✅' if ok else '❌'
    print(f'  {status} {name}')
    if not ok:
        all_pass = False

if all_pass:
    print('\n🎉 All subsystems active and producing valid outputs!')
else:
    print('\n⚠️  Some subsystems may need attention')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 10: Save to HuggingFace (optional)
# ═══════════════════════════════════════════════════════════════════
from huggingface_hub import HfApi, login, create_repo
from safetensors.torch import save_file

HF_REPO = 'Theory903/lasmoid-10m-test'

# Login (skip if already logged in)
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret('HF_TOKEN'))
except:
    try:
        login()
    except:
        print('⚠️  HF login skipped — saving locally only')
        HF_REPO = None

# Save
save_dir = os.path.join(WORK_DIR, 'checkpoints', '10m_test')
os.makedirs(save_dir, exist_ok=True)

save_file(model.state_dict(), os.path.join(save_dir, 'model.safetensors'))
with open(os.path.join(save_dir, 'config.json'), 'w') as f:
    json.dump(cfg, f, indent=2)

readme = f'''---
license: apache-2.0
tags: [lasmoid, test, tiny]
---
# Lasmoid-10M (Test)
Quick validation of the Lasmoid architecture. {n_params:.1f}M params trained on TinyStories.
'''
with open(os.path.join(save_dir, 'README.md'), 'w') as f:
    f.write(readme)

if HF_REPO:
    api = HfApi()
    create_repo(HF_REPO, exist_ok=True)
    api.upload_folder(folder_path=save_dir, repo_id=HF_REPO,
                      commit_message='Lasmoid-10M test checkpoint')
    print(f'\n☁️  Uploaded to https://huggingface.co/{HF_REPO}')
else:
    print(f'\n💾 Saved locally: {save_dir}')

print('\n' + '═' * 60)
print('🏁 DONE! The 10M model validates the full Lasmoid pipeline.')
print('   Next: Train 100M or 300M with the production notebook.')
print('═' * 60)